In [ ]:
import pandas as pd
import sys
sys.path.append('../')

In [ ]:
from constants import brussels_codes_muni_dict

In [ ]:
pop = pd.read_csv('output/workers_with_cars.csv')

# Total stats

In [ ]:
print("=== Global stats ===")
print(f"Overall car ownership rate: {pop['has_car'].mean():.3f}")
print(f"Number of workers with a car: {pop['has_car'].sum()}")

# % workers with car

In [ ]:
pop['municipality'] = pop['assigned_sector'].str[:5]

cars_per_municipality = (
    pop.groupby("municipality")
    .agg(
        n_agents=("has_car", "size"),
        n_cars=("has_car", "sum"),
    )
    .reset_index()
)

cars_per_municipality["pct_with_car"] = (
    100 * cars_per_municipality["n_cars"] / cars_per_municipality["n_agents"]
)

cars_per_municipality["muni_name"] = (
    cars_per_municipality["municipality"]
    .astype(int)
    .map(brussels_codes_muni_dict)
)

cars_per_municipality = cars_per_municipality[
    ["municipality", "muni_name", "n_agents", "n_cars", "pct_with_car"]
]

In [ ]:
print(cars_per_municipality.sort_values("pct_with_car", ascending=False))

# Total number of cars per municipality

In [ ]:
# Load population and cars per statistical sector
sector_stats = pd.read_csv('input_data/cars_per_hh_ss_2021_cleaned.csv')

cars_per_sector = sector_stats.rename(
    columns={
        'CD_SECTOR': 'sector_id',
        'NUM_HOUSEHOLDS': 'n_households',
        'NUM_CARS': 'n_cars',
    }
)

In [ ]:
cars_per_sector['municipality'] = cars_per_sector['sector_id'].str[:5]

statbel_cars_per_municipality = (
    cars_per_sector.groupby("municipality")
    .agg(
        n_households=("n_households", "sum"),
        n_cars=("n_cars", "sum"),
    )
    .reset_index()
)

In [ ]:
cars_per_municipality = cars_per_municipality.merge(
    statbel_cars_per_municipality[["municipality", "n_households", "n_cars"]],
    on="municipality",
    how="left",
    suffixes=("", "_statbel"),
)

In [ ]:
cars_per_municipality["pct_synth_statbel"] = (
    100 * cars_per_municipality["n_cars"] / cars_per_municipality["n_cars_statbel"]
)

In [ ]:
print(cars_per_municipality[['muni_name', 'n_cars', 'n_cars_statbel', 'pct_synth_statbel']])